# 02 - Tiền xử lý & xử lý mất cân bằng (Spark)

Giai đoạn 2-3 của kế hoạch: làm sạch dữ liệu, chuẩn hóa Amount/Time, chia train/test theo thời gian (time-based split), rồi xử lý mất cân bằng (class weighting + SMOTE).

Notebook này gọi trực tiếp các hàm trong `src/spark_etl.py` và `src/imbalance.py`.

In [1]:
import os
import sys

os.environ.setdefault("JAVA_HOME", r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot")
os.environ.setdefault("HADOOP_HOME", r"D:\fraud-detection\hadoop")
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["HADOOP_HOME"] + r"\bin;" + os.environ["PATH"]

sys.path.insert(0, os.path.abspath("../src"))

from config import LABEL_COL, RAW_CSV_PATH
from spark_etl import get_spark_session, read_raw_data, clean_data, scale_features, time_based_split
from imbalance import add_class_weight_column, smote_oversample

spark = get_spark_session("Preprocessing-Notebook")
spark

## Làm sạch dữ liệu

Loại bỏ duplicate và các dòng thiếu đặc trưng/nhãn quan trọng.

In [2]:
df = read_raw_data(spark, RAW_CSV_PATH)
df_clean = clean_data(df)

[Clean] Loại bỏ 1,081 dòng (duplicate/missing). Còn lại 283,726 dòng.


## Chuẩn hóa Amount/Time

`scale_features` tạo thêm 2 cột `Amount_scaled`/`Time_scaled` (StandardScaler, cùng thang đo ~N(0,1) với V1..V28) - dùng để HUẤN LUYỆN mô hình. Cột `Amount`/`Time` gốc được giữ nguyên cho EDA/trực quan hóa.

In [3]:
df_scaled = scale_features(df_clean)
df_scaled.select("Time", "Time_scaled", "Amount", "Amount_scaled", LABEL_COL).show(5)

+----+-------------------+------+--------------------+-----+
|Time|        Time_scaled|Amount|       Amount_scaled|Class|
+----+-------------------+------+--------------------+-----+
|22.0| -1.996356058060835|  0.89| -0.3497719017255733|    0|
|40.0|  -1.99597695943518| 13.84|-0.29805453307185503|    0|
|64.0| -1.995471494600973| 54.99|-0.13371710294826386|    0|
|64.0| -1.995471494600973|  7.55| -0.3231743978465182|    0|
|68.0|-1.9953872504619385|  2.69|-0.34258338716366654|    0|
+----+-------------------+------+--------------------+-----+
only showing top 5 rows


## Time-based split

Chia train/test theo thời gian (không phải random split) - mô phỏng thực tế: mô hình dự đoán giao dịch TƯƠNG LAI dựa trên giao dịch QUÁ KHỨ.

In [4]:
train_df, test_df = time_based_split(df_scaled)

[Split] Ngưỡng thời gian (cutoff): 132816.0s


[Split] Train: 198,402 dòng | Test: 85,324 dòng


## Xử lý mất cân bằng: Class Weighting

Gán trọng số nghịch đảo tần suất lớp - dùng trực tiếp với `weightCol` trong Logistic Regression/Random Forest của Spark MLlib, không cần thay đổi số lượng dòng dữ liệu.

In [5]:
train_weighted_df = add_class_weight_column(train_df)
train_weighted_df.select(LABEL_COL, "class_weight").distinct().show()

[Imbalance] Trọng số lớp (class weight): {1: 271.0409836065574, 0: 0.5009240744107132}


+-----+------------------+
|Class|      class_weight|
+-----+------------------+
|    1| 271.0409836065574|
|    0|0.5009240744107132|
+-----+------------------+



## Xử lý mất cân bằng: SMOTE Oversampling

Lấy mẫu train về pandas rồi oversample lớp thiểu số (gian lận) bằng SMOTE (`imblearn`), vì Spark MLlib chưa hỗ trợ SMOTE trực tiếp. `sampling_strategy=0.1` nghĩa là sau oversample, lớp gian lận bằng 10% lớp bình thường.

In [6]:
balanced_pdf = smote_oversample(train_df)
balanced_pdf[LABEL_COL].value_counts()

[SMOTE] Trước oversample: {0: 198036, 1: 366}


[SMOTE] Sau oversample: {0: 198036, 1: 19803}


Class
0    198036
1     19803
Name: count, dtype: int64

In [7]:
spark.stop()